# analyze_qa_retrieval

각 세션의 `call_6_qa/calls.jsonl`을 읽어 QA마다:
1. `[Retrieved Memory]` ~ `[Recent Conversation]` 사이의 `[<...ago>] <text>` 라인에서 `<text>`만 추출
2. 같은 세션 `memory_snapshots/session_N/graph/graph.json`에서 동일 content를 가진 노드의 `conv_id`, `turn_id`를 매칭
3. 표(`memory_text | node_type | conv | turn`)로 출력
4. question / generated_answer / ground_truth_answer / opposed_implicit_reasoning / retrieved_conv_ids 함께 출력

In [1]:
import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

BASE_DIR = Path('config_0_outputs_Qwen3-1.7B_opposed/session_0_49')
RESULTS_PATH = BASE_DIR / 'results_Qwen3-1.7B_opposed_session_0_49.json'
QA_DATASET_PATH = Path('../dataset/implexconv/ImplexConv_opposed_qa.json')

# Session range to process (inclusive)
START_SESSION = 0
END_SESSION   = 4

# Max chars to show for memory text in the table (None = full)
TEXT_PREVIEW_LEN = None

In [2]:
# Matches: '[1mo ago] text', '[27d ago] text', '  ↳ shifted to: [24d ago] text'
MEM_LINE_RE = re.compile(r'^(?:\s*↳\s*shifted to:\s*)?\[([^\]]*?ago)\]\s+(.+?)\s*$')


def extract_memory_lines(user_prompt: str):
    """Return list of (timestamp_str, text) extracted between [Retrieved Memory] and [Recent Conversation]."""
    try:
        start = user_prompt.index('[Retrieved Memory]')
    except ValueError:
        return []
    try:
        end = user_prompt.index('[Recent Conversation]', start)
    except ValueError:
        end = len(user_prompt)

    block = user_prompt[start:end]
    out = []
    for ln in block.splitlines():
        m = MEM_LINE_RE.match(ln)
        if m:
            out.append((m.group(1), m.group(2)))
    return out


def load_graph(session_num: int) -> dict:
    p = BASE_DIR / 'memory_snapshots' / f'session_{session_num}' / 'graph' / 'graph.json'
    with open(p) as f:
        return json.load(f)


def build_content_index(graph: dict) -> dict:
    """content -> list of (node_type, conv_id, turn_id, node_id)"""
    idx = {}
    for nid, n in graph['nodes'].items():
        c = n.get('content')
        if not isinstance(c, str):
            continue
        idx.setdefault(c, []).append((n['node_type'], n.get('conv_id'), n.get('turn_id'), nid))
    return idx


def lookup_text(index: dict, text: str):
    """Return list of matches; empty list if not found."""
    return index.get(text, [])


def preview(text: str, n=None):
    if n is None or len(text) <= n:
        return text
    return text[:n] + '…'

In [3]:
# Load aux files once: results (for generated/gt answers) and qa dataset (for opposed_implicit_reasoning, retrieved_conv_ids)
with open(RESULTS_PATH) as f:
    RESULTS = json.load(f)
with open(QA_DATASET_PATH) as f:
    QA_DATASET = json.load(f)

RESULTS_BY_SID = {r['session_id']: r for r in RESULTS}
QA_BY_SID      = {d['session_id']: d for d in QA_DATASET}

print(f'results: {len(RESULTS)} sessions, qa_dataset: {len(QA_DATASET)} sessions')

results: 50 sessions, qa_dataset: 1433 sessions


In [4]:
# node_type == 's' 인 행만 출력
for sid in range(START_SESSION, END_SESSION + 1):
    prompt_dir = BASE_DIR / 'prompt_log' / f'session_{sid}'
    if not prompt_dir.exists():
        print(f'[session {sid}] prompt_log 없음, skip')
        continue

    matches = [d for d in prompt_dir.iterdir() if d.name.startswith('call_6_')]
    if not matches:
        print(f'[session {sid}] call_6_* 없음, skip')
        continue
    calls_path = sorted(matches)[0] / 'calls.jsonl'
    if not calls_path.exists():
        print(f'[session {sid}] calls.jsonl 없음, skip')
        continue

    with open(calls_path) as f:
        records = [json.loads(line) for line in f if line.strip()]

    try:
        graph = load_graph(sid)
    except FileNotFoundError:
        print(f'[session {sid}] graph.json 없음, skip')
        continue
    content_idx = build_content_index(graph)

    session_result = RESULTS_BY_SID.get(sid, {})
    qa_results = session_result.get('qa_results', [])
    qa_list    = QA_BY_SID.get(sid, {}).get('qa', [])

    print('#' * 100)
    print(f'#  SESSION {sid}    (records: {len(records)}, qa_results: {len(qa_results)})  [node_type == s only]')
    print('#' * 100)

    for i, rec in enumerate(records):
        print('=' * 100)
        print(f'[ session {sid} / QA #{i} ]')

        # ---- QA meta ----
        if i < len(qa_results):
            qr = qa_results[i]
            q = qr['question']
            reasoning = next(
                (qa.get('opposed_implicit_reasoning') for qa in qa_list if qa['question'] == q),
                None,
            )
            retrieved_conv_ids = next(
                (qa.get('retrieved_conv_ids') for qa in qa_list if qa['question'] == q),
                None,
            )
            print(f'\n■ question\n{q}')
            print(f'\n■ generated_answer\n{qr["generated_answer"]}')
            print(f'\n■ ground_truth_answer\n{qr["ground_truth_answer"]}')
            print(f'\n■ opposed_implicit_reasoning\n{reasoning if reasoning is not None else "(매칭 없음)"}')
            print(f'\n■ retrieved_conv_ids: {retrieved_conv_ids if retrieved_conv_ids is not None else "(매칭 없음)"}')
        else:
            print(f'(qa_results[{i}] 없음)')

        # ---- extract memory lines & lookup (node_type == 's' only) ----
        user_prompt = rec.get('user_prompt', '')
        mem_lines = extract_memory_lines(user_prompt)

        rows = []
        for ts, text in mem_lines:
            hits = lookup_text(content_idx, text)
            for nt, cid, tid, _nid in hits:
                if nt != 's':
                    continue
                rows.append({
                    'timestamp':   ts,
                    'memory_text': preview(text, TEXT_PREVIEW_LEN),
                    'node_type':   nt,
                    'conv':        cid,
                    'turn':        tid,
                })

        print(f'\n■ retrieved memory → graph lookup (s only: {len(rows)} rows / total mem lines: {len(mem_lines)})')
        if rows:
            df = pd.DataFrame(rows, columns=['timestamp', 'memory_text', 'node_type', 'conv', 'turn'])
            with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
                display(df)
        else:
            print('(node_type == s 인 항목 없음)')


####################################################################################################
#  SESSION 0    (records: 5, qa_results: 5)  [node_type == s only]
####################################################################################################
[ session 0 / QA #0 ]

■ question
How can I find reliable tech bloggers to recommend educational apps for my children?

■ generated_answer
To find reliable tech bloggers for educational apps, follow these steps: 1. Identify bloggers specializing in educational technology, such as Edutopia, Common Sense Education, or School Two. 2. Check their credibility by reviewing their content, author bios, and previous work. 3. Look for blogs with consistent, positive feedback and a history of trustworthy recommendations. 4. Engage with their community and ask for recommendations. 5. Cross-reference multiple sources to ensure reliability.

■ ground_truth_answer
Finding reliable tech bloggers might be challenging, especially while sup

,timestamp,memory_text,node_type,conv,turn
0,1mo ago,The user is looking for high-quality educational technology tools for their children and is concerned about trustworthiness of recommendations.,s,3,8
1,1mo ago,The user is interested in using educational apps in their class.,s,12,2
2,1mo ago,The user is considering starting their own channel and sharing their thoughts and insights with others.,s,48,12
3,21d ago,The user prefers outdoor activities and is looking for a competitive hobby.,s,66,6
4,6d ago,The user is passionate about swimming and has been swimming since they were 6 years old.,s,97,2
5,10d ago,"The user is looking for a gym with a variety of weight machines, free weights, treadmills, and elliptical machines.",s,89,6
6,1mo ago,"The user is seeking educational apps for their kids that are both fun and effective, and they are looking for reliable sources to find the best options.",s,2,6
7,1mo ago,The user frequently visits popular tech blogs to stay updated on the latest educational apps and their features.,s,4,10
8,1mo ago,The user is seeking recommendations for educational apps suitable for their children's interests and grade levels.,s,5,6
9,1mo ago,The user feels more confident now in finding the best educational apps for their kids.,s,0,14


[ session 0 / QA #1 ]

■ question
What wrestling moves should I master to become a successful wrestler like my idol John Cena?

■ generated_answer
To master the moves of John Cena, focus on techniques like the *Cena Press* (a powerful uppercut and shoulder press), the *Cena Drop* (a devastating armbar), and the *Cena Backdrop* (a signature back body drop). Emphasize footwork, stamina, and mental resilience. Study his signature moves in context, practice under a coach, and stay committed to consistent training. Inspiration from Cena's charisma and strategic mindset will help you excel in the ring.

■ ground_truth_answer
While mastering wrestling moves is crucial to becoming a successful wrestler like John Cena, it's important to consider that a medical condition, such as epilepsy, makes it unsafe to participate in contact sports. Therefore, pursuing wrestling may not be advisable in your situation. Focus on alternative ways to engage with wrestling, such as coaching or commentary.

■ op

,timestamp,memory_text,node_type,conv,turn
0,1mo ago,"The user is a huge fan of wrestling, specifically John Cena, and follows it regularly.",s,29,10
1,28d ago,The user is excited about the wrestling match tonight and believes John Cena will win the championship belt.,s,53,0
2,1mo ago,The user is discussing a wrestling match between John Cena and Randy Orton.,s,29,2
3,1mo ago,The user is a huge fan of wrestling and watches it on TV every week.,s,29,6
4,27d ago,The user loves watching live matches and is excited about the wrestling match tonight.,s,54,2
5,29d ago,"The user likes Becky Lynch and The Rock, and is passionate about wrestling.",s,51,4
6,27d ago,The user is interested in wrestling and shows confidence in John Cena's abilities.,s,55,0
7,1mo ago,The user loves John Cena and considers him a talented wrestler.,s,31,4
8,27d ago,The user loves wrestling and is a huge fan of John Cena.,s,55,2
9,1mo ago,The user is interested in starting with basic wrestling moves and training together.,s,32,12


[ session 0 / QA #2 ]

■ question
How can I attend a WWE match featuring Canadian wrestlers like Edge and Trish Stratus this year?

■ generated_answer
To attend a WWE match featuring Canadian wrestlers like Edge and Trish Stratus, check WWE's official website for upcoming events. Tickets are available for WWE pay-per-view events, such as WrestleMania, and can be purchased through official ticketing platforms. Ensure to check the date and location of the event, and consider attending live events to experience the energy and atmosphere of wrestling.

■ ground_truth_answer
Attending a WWE match featuring Canadian wrestlers like Edge and Trish Stratus this year may be challenging, especially since their monthly budget was reduced, forcing them to cut back on discretionary spending. This financial constraint could limit options for tickets, travel, and related expenses, making it difficult to attend the event.

■ opposed_implicit_reasoning
Their monthly budget was reduced, forcing them to c

...

,timestamp,memory_text,node_type,conv,turn
0,27d ago,"The user is interested in Korean films, specifically 'Parasite', which is a social satire exploring class dynamics and a thriller.",s,54,2
1,18d ago,The user values storytelling techniques in explaining complex topics.,s,72,6
2,29d ago,The user values complex and multi-dimensional characters with strong motivations and backstories.,s,50,6
3,1mo ago,The user likes to thoroughly analyze every book they read to understand the message the author is trying to convey.,s,11,4
4,10d ago,The user feels stuck in a situation where the project is consuming their every waking moment and they don't know how to make time for themselves without sacrificing the project.,s,89,14
5,10d ago,The user is feeling overwhelmed by a project that has become an all-encompassing thing for them.,s,89,10
6,29d ago,"The user loves Korean dramas for their ability to weave together romance, drama, and comedy and how they can make them laugh and cry in the same episode.",s,50,2
7,1mo ago,The user is interested in exploring Korean dramas and wants to know what makes them special.,s,44,8
8,1mo ago,The user is interested in exploring Korean dramas and wants to know what makes them special.,s,44,10
9,1mo ago,The user has a soft spot for Korean dramas,s,48,6


[ session 4 / QA #3 ]

■ question
How can I simplify complex medical concepts without confusing my audience when explaining them?

■ generated_answer
Use relatable analogies, storytelling, and visual aids to simplify complex medical concepts. Break down technical terms into understandable language, and emphasize key points with clear, engaging narratives. Prioritize clarity and relevance, ensuring your audience connects with the material through personal experiences or familiar scenarios.

■ ground_truth_answer
To simplify complex medical concepts effectively, focus on one topic at a time. Use analogies and relatable examples, but be aware that the need to cover a broad range of topics has led to shallow explanations that lack depth and clarity, making it challenging to convey nuanced understanding. Prioritize clarity over breadth to avoid confusion.

■ opposed_implicit_reasoning
The need to cover a broad range of topics has led to shallow explanations that lack depth and clarity.

■ r

,timestamp,memory_text,node_type,conv,turn
0,19d ago,The user wants to use relatable analogies to explain the immune system's response to an infection.,s,70,10
1,18d ago,The user values storytelling techniques in explaining complex topics.,s,72,6
2,19d ago,"The user wants to use a narrative to explain the immune system's response to a virus, emphasizing relatability and engagement.",s,70,12
3,20d ago,The user believes that engagement is crucial for effective communication of scientific information.,s,69,8
4,1mo ago,The user likes to thoroughly analyze every book they read to understand the message the author is trying to convey.,s,11,4
5,10d ago,The user is feeling overwhelmed by a project that has become an all-encompassing thing for them.,s,89,10
6,19d ago,The user is preparing for a medical presentation and is seeking advice on making complex concepts engaging.,s,70,0
7,21d ago,The user appreciates the science communicator's ability to simplify complex medical concepts.,s,66,10
8,19d ago,The user wants to make complex medical concepts relatable to their audience through analogies.,s,70,6
9,21d ago,The user is interested in understanding complex medical concepts and appreciates effective communication of technical information.,s,66,6


[ session 4 / QA #4 ]

■ question
How can I find time to create a vintage music playlist with songs from the 60s to 80s era?

■ generated_answer
To create a vintage music playlist from 60s to 80s, start by organizing your music library and identifying songs from those eras. Use streaming services or vinyl records to access classic tracks. Schedule time each week for curating and listening to music. Incorporate diverse genres and artists to meet your preference for a mix of popular and timeless songs. Consider using playlists with upbeat tempo and positive lyrics for a relaxing experience. Enjoy the nostalgic journey of revisiting classic music.

■ ground_truth_answer
Given that you've been volunteering or working on a project that requires your full attention, it might be challenging to find time for leisure activities, including creating a vintage music playlist from the 60s to 80s. Consider scheduling small blocks of time when possible, but be realistic about your current commitments

,timestamp,memory_text,node_type,conv,turn
0,7d ago,The user is looking for a new record player to enhance sound quality.,s,94,6
1,27d ago,"The user is interested in Korean films, specifically 'Parasite', which is a social satire exploring class dynamics and a thriller.",s,54,2
2,10d ago,The user feels stuck in a situation where the project is consuming their every waking moment and they don't know how to make time for themselves without sacrificing the project.,s,89,14
3,10d ago,The user is feeling overwhelmed by a project that has become an all-encompassing thing for them.,s,89,10
4,29d ago,The user values complex and multi-dimensional characters with strong motivations and backstories.,s,50,6
5,7d ago,The user wants to invite classmates to an event and make it a whole event out of it.,s,95,10
6,10d ago,The user is a huge fan of vintage music and is exploring it from the 50s to the 70s.,s,88,10
7,4d ago,The user wants a playlist of five songs for a birthday gift to a friend born in 1965.,s,101,4
8,10d ago,The user is feeling nostalgic and recalling old music.,s,88,2
9,6d ago,The user expresses interest in the music collection and mentions enjoying vintage vinyl records.,s,97,2


In [ ]:
for sid in range(START_SESSION, END_SESSION + 1):
    prompt_dir = BASE_DIR / 'prompt_log' / f'session_{sid}'
    if not prompt_dir.exists():
        print(f'[session {sid}] prompt_log 없음, skip')
        continue

    matches = [d for d in prompt_dir.iterdir() if d.name.startswith('call_6_')]
    if not matches:
        print(f'[session {sid}] call_6_* 없음, skip')
        continue
    calls_path = sorted(matches)[0] / 'calls.jsonl'
    if not calls_path.exists():
        print(f'[session {sid}] calls.jsonl 없음, skip')
        continue

    with open(calls_path) as f:
        records = [json.loads(line) for line in f if line.strip()]

    try:
        graph = load_graph(sid)
    except FileNotFoundError:
        print(f'[session {sid}] graph.json 없음, skip')
        continue
    content_idx = build_content_index(graph)

    session_result = RESULTS_BY_SID.get(sid, {})
    qa_results = session_result.get('qa_results', [])
    qa_list    = QA_BY_SID.get(sid, {}).get('qa', [])

    print('#' * 100)
    print(f'#  SESSION {sid}    (records: {len(records)}, qa_results: {len(qa_results)})')
    print('#' * 100)

    for i, rec in enumerate(records):
        print('=' * 100)
        print(f'[ session {sid} / QA #{i} ]')

        # ---- QA meta ----
        if i < len(qa_results):
            qr = qa_results[i]
            q = qr['question']
            reasoning = next(
                (qa.get('opposed_implicit_reasoning') for qa in qa_list if qa['question'] == q),
                None,
            )
            retrieved_conv_ids = next(
                (qa.get('retrieved_conv_ids') for qa in qa_list if qa['question'] == q),
                None,
            )
            print(f'\n■ question\n{q}')
            print(f'\n■ generated_answer\n{qr["generated_answer"]}')
            print(f'\n■ ground_truth_answer\n{qr["ground_truth_answer"]}')
            print(f'\n■ opposed_implicit_reasoning\n{reasoning if reasoning is not None else "(매칭 없음)"}')
            print(f'\n■ retrieved_conv_ids: {retrieved_conv_ids if retrieved_conv_ids is not None else "(매칭 없음)"}')
        else:
            print(f'(qa_results[{i}] 없음)')

        # ---- extract memory lines & lookup ----
        user_prompt = rec.get('user_prompt', '')
        mem_lines = extract_memory_lines(user_prompt)

        rows = []
        for ts, text in mem_lines:
            hits = lookup_text(content_idx, text)
            if not hits:
                rows.append({
                    'timestamp':   ts,
                    'memory_text': preview(text, TEXT_PREVIEW_LEN),
                    'node_type':   '(미매칭)',
                    'conv':        None,
                    'turn':        None,
                })
            else:
                for nt, cid, tid, _nid in hits:
                    rows.append({
                        'timestamp':   ts,
                        'memory_text': preview(text, TEXT_PREVIEW_LEN),
                        'node_type':   nt,
                        'conv':        cid,
                        'turn':        tid,
                    })

        print(f'\n■ retrieved memory → graph lookup ({len(mem_lines)} lines, {sum(1 for r in rows if r["conv"] is None)} 미매칭)')
        if rows:
            df = pd.DataFrame(rows, columns=['timestamp', 'memory_text', 'node_type', 'conv', 'turn'])
            with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
                display(df)
        else:
            print('(추출된 메모리 라인 없음)')
